# 01 — Chat Models (Text → Text)
### LangChain Foundations · Part 1 of 2

Every LLM app starts the same place: getting a model to answer you. This notebook covers
that one skill — properly — across **every provider you actually have access to**:

| Provider | Your access | Why it's here |
|---|---|---|
| **OpenAI** | Paid plan | Industry default, best docs, most reliable |
| **Google Gemini** | Free tier | Generous free quota, huge context window |
| **Groq** | Free tier | Insanely fast inference (300–600+ tokens/sec) on open models |
| **Anthropic (Claude)** | ❌ No key — code shown as reference | So you know the pattern for when you do get access |
| **Hugging Face** | Free (open-source) | Fully free, no billing ever, runs open models |

> 📌 **What's new (as of late 2026)** — a few things worth knowing before you write a line of code:
> - LangChain moved to **1.x** in late 2025. The core got simpler; **LangGraph is now the official way to build agents** (you'll get there in this repo too).
> - The old "LLM" (raw text-completion) classes are basically legacy now. Every model you'll use is a **chat model** — it takes a *list of messages* and returns a *message*, not a raw string.
> - OpenAI's GPT-5.x line no longer uses `temperature` the same way older GPT-4 models did — you'll see a note on this below.
> - Groq's free tier (no credit card) currently covers Llama 3.1/3.3, Llama 4 Scout, Qwen3, and OpenAI's own open-weight `gpt-oss` models — all at genuinely fast, genuinely free rate limits.
> - Google's free tier is now **Flash-only** (`gemini-2.5-flash` and lite variants) — the Pro models require billing.

---

## The one idea that makes all of this click

LangChain's entire value proposition for models is: **write your code once, swap the provider by changing one line.**

Every chat model in LangChain — no matter who makes it — exposes the *same* four methods:

| Method | What it does |
|---|---|
| `.invoke(messages)` | Send messages, get one complete response back |
| `.stream(messages)` | Get the response back token-by-token, as it's generated |
| `.batch([messages1, messages2, ...])` | Send many requests at once, efficiently |
| `.ainvoke(messages)` | The async version of `.invoke()` |

You'll see this over and over below: the *setup* line differs per provider (different import, different class name), but every call after that looks identical. That consistency is the whole point.

## Setup

Install what you need (uncomment the line below the first time). You only need the packages
for providers you're actually going to use.

In [1]:
# Run this once. Safe to re-run — pip will just confirm things are already installed.
# %pip install -q langchain langchain-core python-dotenv
# %pip install -q langchain-openai              # OpenAI
# %pip install -q langchain-google-genai        # Gemini
# %pip install -q langchain-groq                # Groq
# %pip install -q langchain-anthropic           # Anthropic (reference only — needs a key you don't have)
# %pip install -q langchain-huggingface huggingface_hub  # Hugging Face


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads a .env file sitting next to this notebook

# .env should look like this (fill in only what you have):
#
# OPENAI_API_KEY=sk-...
# GOOGLE_API_KEY=...
# GROQ_API_KEY=gsk_...
# ANTHROPIC_API_KEY=...        # you don't have this one — leave it out, that's fine
# HUGGINGFACEHUB_API_TOKEN=hf_...

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("Environment ready.")


Environment ready.


## Messages — the universal input format

Before touching any provider, understand what you're actually sending. Every chat model call
is a **list of message objects**, each with a role:

| Message type | Role | Purpose |
|---|---|---|
| `SystemMessage` | Sets the model's behavior/persona | "You are a concise, formal support agent." |
| `HumanMessage` | What the user said | "My order hasn't arrived yet." |
| `AIMessage` | What the model said (used when building up chat history) | "Sorry to hear that — can you share your order ID?" |

This is covered in depth in `02_Prompts/`, but you need the basic shape now to actually call a model.

In [3]:
messages = [
    SystemMessage(content="You are a friendly, concise customer support agent for a phone company."),
    HumanMessage(content="My internet has been down since this morning."),
]
# We'll actually send this to a real model in the next section.


---
## 1. OpenAI — `ChatOpenAI`

You have a paid plan here, so this is the one you can lean on hardest. Current line-up (2026):
`gpt-5.1` / `gpt-5.2` (frontier), `gpt-5-mini` (cheap + very capable — good default for learning),
`gpt-5-nano` (cheapest, simplest tasks).

**Important GPT-5 quirk:** unlike GPT-4, `temperature` isn't really a knob GPT-5 respects the old
way — instead you get `reasoning_effort` (`"minimal"` / `"low"` / `"medium"` / `"high"`) to control
how much the model "thinks" before answering. Keep `reasoning_effort="minimal"` for simple chat —
it's faster and cheaper.

In [4]:
from langchain_openai import ChatOpenAI

openai_model = ChatOpenAI(
    model="gpt-5-mini",
    reasoning_effort="minimal",   # GPT-5 series: "minimal" is fastest/cheapest for simple chat
)

response = openai_model.invoke(messages)
print(response.content)


Sorry about that — I can help. A few quick checks will narrow it down.

1) Are you seeing any lights on your modem/router? (Which are solid vs blinking?)
2) Is the outage affecting wired devices, Wi‑Fi, or both?
3) Have you already tried power-cycling the modem/router (unplug 30 seconds, plug back in)?
4) Do you know if neighbors or others in your building are affected?

If you prefer, give me your account phone number or service address and I can run a line test and check for outages in your area. I won’t perform any changes without your permission.


In [5]:
# Streaming — watch the response arrive token by token instead of waiting for the whole thing.
for chunk in openai_model.stream(messages):
    print(chunk.content, end="", flush=True)


Sorry about that — I can help. A few quick questions so I can diagnose and escalate if needed:

1. Are you on a wired (Ethernet) connection or Wi‑Fi?
2. Do any lights on your modem/router look red or off? If so, which lights?
3. Have you tried restarting the modem/router? If not, please power it off, wait 30 seconds, then power it on and tell me whether anything changed.
4. Is this affecting one device or all devices in your home?
5. Your account name or phone number on the account (or the last 4 digits) so I can check outage reports.

If you’d rather not share account info here, I can give general troubleshooting steps next.

---
## 2. Google Gemini — `ChatGoogleGenerativeAI`

Free tier as of 2026 is **Flash-only**: `gemini-2.5-flash` (stable, the safe daily driver) or
`gemini-2.5-flash-lite` (higher rate limit, lower capability — good for simple/high-volume tasks).
The Pro models and the newer Gemini 3 preview line require billing enabled.

Notice the code shape below is **identical** to OpenAI — same `messages` list, same `.invoke()`,
same `.stream()`. Only the import and class name changed.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
)

response = gemini_model.invoke(messages)
print(response.content)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


I'm sorry to hear your internet has been down. Let's get that fixed for you.

To start, could you please provide your account number or the phone number associated with your account?


In [7]:
for chunk in gemini_model.stream(messages):
    print(chunk.content, end="", flush=True)


I'm sorry to hear your internet has been down all morning. Let's get this fixed for you.

Could you please provide your account number or the phone number associated with your service?

---
## 3. Groq — `ChatGroq`

Groq doesn't train its own models — it runs *open-weight* models (Llama, Qwen, OpenAI's `gpt-oss`)
on custom LPU hardware that's dramatically faster than normal GPU inference. Free tier: no credit
card, ~30 requests/min, and it's shared across whichever model you pick.

Good free-tier model choices right now:

| Model string | Best for |
|---|---|
| `llama-3.1-8b-instant` | Simple, high-volume tasks — extremely fast |
| `llama-3.3-70b-versatile` | General-purpose, noticeably smarter, still fast |
| `openai/gpt-oss-120b` | OpenAI's own open-weight model, strong reasoning |

In [8]:
from langchain_groq import ChatGroq

groq_model = ChatGroq(
    model="qwen/qwen3.8-27b"
    #temperature=0.7,
)

import time
start = time.time()
response = groq_model.invoke(messages)
print(response.content)
print(f"\n--- {time.time() - start:.2f}s (compare this to how long OpenAI/Gemini took above) ---")


I’m sorry to hear that. Let’s try to get your internet back up right away.

Could you please check:
1. Is the **Internet (WAN/Online)** light on your router solid green or blinking?
2. Has anyone else in your home tried to reset the router (unplugged it for 10 seconds and plugged it back in)?

If the light is solid green but you still have no internet, please let me know, and we can take the next steps.

--- 0.84s (compare this to how long OpenAI/Gemini took above) ---


---
## 4. Anthropic (Claude) — `ChatAnthropic`  *(reference only — you don't have this key)*

Included so the pattern is here when you get access — notice it's **exactly the same shape** as
every provider above. That consistency is not an accident; it's the entire design of LangChain's
model interface.

In [9]:
from langchain_anthropic import ChatAnthropic

# This cell will raise an authentication error if you run it without an ANTHROPIC_API_KEY —
# that's expected. It's here to show you the pattern, not to be run today.

claude_model = ChatAnthropic(
    model="claude-sonnet-5",
    temperature=0.7,
)

#response = claude_model.invoke(messages)
#print(response.content)


---
## 5. Hugging Face — free, open-source models

Two very different ways to use Hugging Face, and it matters which you pick:

| Approach | Cost | Where it runs | Setup |
|---|---|---|---|
| **Serverless Inference API** (`HuggingFaceEndpoint`) | Free (rate-limited) | HF's servers | Just need a free HF account + token |
| **Fully local** (`transformers` pipeline) | Free, unlimited | Your own machine | No token needed, but you download the model weights (can be GBs) and need decent CPU/GPU |

For learning, start with the serverless endpoint — it's the closest experience to the API-based
providers above.

In [10]:
print("Api Token:", bool(os.getenv("HUGGINGFACEHUB_API_TOKEN")))
print("Hub Token:", bool(os.getenv("HUGGINGFACEHUB_HUB_TOKEN")))

Api Token: True
Hub Token: False


In [11]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    #huggingfacehub_hub_token=os.getenv("HUGGINGFACEHUB_HUB_TOKEN"),
)

hf_model = ChatHuggingFace(llm=llm_endpoint)

response = hf_model.invoke(messages)
print(response.content)

I'm sorry to hear that you're experiencing issues with your internet. Let's try to get it sorted out. Could you please check if the modem and router are powered on and if all the cables are securely connected? If you've already done this, let me know, and we can proceed from there.


In [12]:
# --- Fully local alternative (no token, no internet call after the first download) ---
#
# from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
# from transformers import pipeline
#
# pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=256)
# local_llm = HuggingFacePipeline(pipeline=pipe)
# local_chat_model = ChatHuggingFace(llm=local_llm)
# local_chat_model.invoke(messages)


---
## The payoff: a provider-agnostic factory function

This is the pattern that makes everything above worth it. Instead of hardcoding one provider
everywhere in your app, write **one function** that hands back whichever model you ask for.
Swapping providers — or falling back when one is down or rate-limited — becomes a one-line change.

In [13]:
def get_chat_model(provider: str, **kwargs):
    """Return a LangChain chat model for the given provider.
    Every model returned here supports the exact same .invoke() / .stream() interface.
    """
    provider = provider.lower()

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=kwargs.get("model", "gpt-5-mini"), reasoning_effort="minimal")

    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=kwargs.get("model", "gemini-2.5-flash"), temperature=0.7)

    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=kwargs.get("model", "llama-3.3-70b-versatile"), temperature=0.7)

    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=kwargs.get("model", "claude-sonnet-5"), temperature=0.7)

    raise ValueError(f"Unknown provider: {provider}")


# Same prompt, three different engines under the hood — the calling code never changes.
for name in ["groq", "gemini", "openai"]:
    try:
        model = get_chat_model(name)
        print(f"--- {name} ---")
        print(model.invoke(messages).content[:200], "...\n")
    except Exception as e:
        print(f"--- {name} failed: {e} ---\n")


--- groq ---
--- groq failed: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}} ---

--- gemini ---
I'm sorry to hear your internet has been down. I can help with that. Could I please get your account number or the phone number associated with your service? ...

--- openai ---
Sorry about that — I can help. A few quick questions to diagnose and get this fixed:

1. Are all devices (phone, laptop, smart TV) unable to connect, or only one device?
2. Are there any lights blinki ...



---
## Generation parameters — the knobs you'll actually touch

| Parameter | What it controls | Analogy |
|---|---|---|
| `temperature` (0–1, sometimes up to 2) | Randomness/creativity of output | 0 = same answer every time (a calculator); 1 = a different, more surprising answer each time (a brainstorm partner) |
| `max_tokens` | Hard cap on response length | A word limit you set before someone starts talking |
| `top_p` | Alternative way to control randomness (nucleus sampling) | Usually leave this alone — pick *either* `temperature` or `top_p`, not both |
| `stop` | Strings that immediately end generation | "Stop talking the moment you type `###`" |
| `streaming` | Whether tokens arrive incrementally | The difference between watching someone type live vs. getting a text message all at once |

**Rule of thumb:** `temperature=0` for anything factual, classification, or code. `temperature=0.7–0.9`
for creative writing, brainstorming, or conversational chat.

---
## Real-world capstone: Multi-provider support assistant with automatic fallback

This is a pattern you'll actually ship in production: **never depend on one provider**. If Groq is
rate-limited, or Gemini's free tier is exhausted for the day, your app should quietly fall back to
the next option instead of crashing in front of a user.

The order below is deliberate: **Groq first** (free + fastest), **Gemini second** (free + generous
context), **OpenAI last** (costs money, so it's the safety net, not the default).

In [14]:
def get_support_reply(ticket_text: str, providers=("groq", "gemini", "openai")) -> tuple[str, str]:
    """Try each provider in order until one succeeds. Returns (reply, provider_used)."""
    system = SystemMessage(content=(
        "You are a calm, helpful customer support agent. Acknowledge the issue, "
        "ask exactly one clarifying question if needed, and keep the reply under 80 words."
    ))
    convo = [system, HumanMessage(content=ticket_text)]

    for provider in providers:
        try:
            model = get_chat_model(provider)
            reply = model.invoke(convo).content
            return reply, provider
        except Exception as e:
            print(f"[{provider} unavailable: {e}] — trying next provider...")
            continue

    raise RuntimeError("All providers failed — check your API keys / network.")


ticket = "I was charged twice for my subscription this month and I need a refund."
reply, used = get_support_reply(ticket)
print(f"Answered by: {used}\n\n{reply}")


[groq unavailable: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}] — trying next provider...
Answered by: gemini

I'm sorry to hear you've been charged twice for your subscription this month. I can certainly help you with this.

To locate your account and process your refund, could you please provide the email address associated with your subscription or the transaction IDs for the charges?


### Try it yourself
1. Add a fourth ticket about a different issue (late delivery, login problem, billing question) and run it through `get_support_reply`.
2. Break the Groq API key on purpose (rename the env var) and confirm the function *actually* falls back to Gemini.
3. Change `get_support_reply`'s `providers` order to put `openai` first — notice you're now paying for every single call, even when a free provider would've worked. That's the real-world cost of getting fallback order wrong.
4. Add Hugging Face as a fourth, fully-free fallback option at the end of the chain.

---
**Next:** `02_Prompts/` — turning that hardcoded `SystemMessage`/`HumanMessage` pattern into reusable, parameterized templates.
